In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import jax.numpy as jnp

import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['text.usetex'] = True
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.grid'] = True
mpl.rcParams['axes.xmargin'] = 0
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['legend.frameon'] = False
mpl.rcParams['savefig.bbox'] = 'tight'

import jaxsp as jsp

from scipy import constants as const

import matplotlib.animation as animation
from IPython.display import HTML

import Stellar_sim_funcs as SSF
import importlib
importlib.reload(SSF)


from scipy.interpolate import interp1d

from collections import defaultdict

from jaxsp.constants import h, om, hbar, Msun, GN, c, m22

from collections import defaultdict


In [ ]:
m22 = 1
u = jsp.set_schroedinger_units(m22)

G = GN.value * (u.from_cm**3) / (u.from_g * u.from_s**2)

In [ ]:
rho = jax.vmap(jsp.rho, in_axes=(0,None))

# rho is a function we can input array r and parameter values to get the density

In [ ]:
cNFWtides_params = jnp.array([
    357964808.148399 * u.from_Msun, 
    25.690207, 
    0.407461, 
    0.012670 * u.from_Kpc, 
    1.857991 * u.from_Kpc, 
    3.729259
])
density_params = jsp.init_core_NFW_tides_params_from_sample(cNFWtides_params)
r99 = jsp.enclosing_radius(0.99, density_params) # radius that encloses 99% of mass
print(r99, "kpc")
density_params

In [ ]:
fig, ax = plt.subplots()
ax.xaxis.set_tick_params(labelsize=15)
ax.yaxis.set_tick_params(labelsize=15)
ax.set_xscale("log")
ax.set_yscale("log")
r = jnp.logspace(jnp.log10(100 * u.from_pc), jnp.log10(r99), 200)
ax.plot(r * u.to_Kpc, rho(r, density_params) * u.to_Msun/u.to_Kpc**3)
ax.set_ylabel(r"$\rho \;\;\mathrm{[M_\odot\;kpc^{-3}]}$", fontsize = 18)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 18)
ax.set_title('DM density profile: NFW with cored center and tidal stripping effects', fontsize = 16)
plt.show()


In [ ]:
potential = jax.vmap(jsp.potential, in_axes=(0,None))

In [ ]:
rmin = .1 * u.from_pc
rmax = jsp.enclosing_radius(0.999, density_params)
N = 512
potential_params = jsp.init_potential_params(density_params, rmin, rmax, N) 
potential_params

In [ ]:
fig, ax = plt.subplots()
ax.set_xscale("log")
ax.xaxis.set_tick_params(labelsize=15)
ax.yaxis.set_tick_params(labelsize=15)
r = jnp.logspace(jnp.log10(rmin), jnp.log10(rmax), 200)
ax.plot(r * u.to_Kpc, potential(r, potential_params) * u.to_kms**2)
ax.set_ylabel(r"$V \;\;\mathrm{[km^2 \;s^{-2}]}$", fontsize = 15)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 15)

ax.set_title('Gravitational Potential Profile', fontsize = 20)
plt.show()

In [ ]:
eval_library = jax.vmap(jax.vmap(jsp.eval_radial_eigenmode, in_axes=(None, 0)), in_axes=(0,None))

In [ ]:
rmin = .1 * u.from_pc
rmax = jsp.enclosing_radius(0.99, density_params)
print(rmax * u.to_Kpc)
N = 1024
a = 1
b = 10
eigenstate_lib = jsp.init_eigenstate_library(potential_params, rmin, rmax, a, b, N)
eigenstate_lib
n = eigenstate_lib.radial_eigenmode_params.n 
l = eigenstate_lib.radial_eigenmode_params.l
# print(l)
# print(len(n))

In [ ]:
fig, ax = plt.subplots()
ax.set_xscale("log")
ax.set_yscale("log")
r = jnp.logspace(jnp.log10(20 * u.from_pc), jnp.log10(rmax), 1000)
R_j_r = eval_library(r, eigenstate_lib.radial_eigenmode_params)
ax.set_title('Radial Eigenmodes', fontsize = 20)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 15)
ax.set_ylabel(r"$R_{j}(r)^2 \;\;\mathrm{[kpc^{-1}]}$", fontsize = 15)
ax.plot(r * u.to_Kpc, R_j_r**2, lw=0.5)
ax.set_ylim([1e-15, 1e5])
plt.show()

In [ ]:
rho_psi = jax.vmap(jsp.rho_psi, in_axes=(0,None,None))

In [ ]:
tol = 1e-7
rmin = 20 * u.from_pc
rmax = jsp.enclosing_radius(0.99, density_params)
wavefunction_params = jsp.init_wavefunction_params(eigenstate_lib, density_params, rmin, rmax, tol)
wavefunction_params

In [ ]:
fig, ax = plt.subplots()
ax.set_xscale("log")
ax.set_yscale("log")

r = jnp.logspace(jnp.log10(rmin), jnp.log10(rmax), 1000)
ax.plot(r * u.to_Kpc, rho(r, density_params) * u.to_Msun / u.to_Kpc**3, label='cNFWtides density')
ax.plot(r * u.to_Kpc, rho_psi(r, wavefunction_params, eigenstate_lib) * u.to_Msun / u.to_Kpc**3, label='wavefunction reconstructed density', ls='--')
ax.set_ylabel(r"$\rho \;\;\mathrm{[M_\odot\;kpc^{-3}]}$", fontsize = 15)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 15)
plt.legend(fontsize = 14)
plt.show()

In [ ]:


rho_psi_vals = rho_psi(r, wavefunction_params, eigenstate_lib)

Phi_psi = SSF.Obtain_pot(rmin, rmax, rho_psi_vals, r)

fig, ax = plt.subplots()
ax.set_xscale("log")
ax.plot(r * u.to_Kpc, Phi_psi * u.to_kms**2, label='wavefunction potential', lw=2, ls='--')
ax.plot(r * u.to_Kpc, potential(r, potential_params) * u.to_kms**2, label='input potential', lw=2)
ax.set_ylabel(r"$V \;\;\mathrm{[km^2 \;s^{-2}]}$", fontsize = 15)
ax.set_xlabel(r"$r \;\;\mathrm{[kpc]}$", fontsize = 15)
plt.legend(fontsize = 14)
plt.show()



In [ ]:

r_orbit = 0.19 * u.from_Kpc

init_pos = np.array([r_orbit, 0, 0]) #Starting as position (r_vir, 0, 0) = (x, y, z)

#at r_orbit, need to find value of the force from the wavefunction potential

acc_mag = SSF.Find_acc_mag_from_Phi(r, Phi_psi, r_orbit)

print(acc_mag)

init_vel = np.sqrt(acc_mag * r_orbit) * np.array([0, 1, 0]) #Circular orbit velocity (vx, vy, vx)

init_pos_sph = SSF.Cartesian_to_sph(init_pos[0], init_pos[1], init_pos[2])

init_vel_sph = SSF.Cartesian_to_sph_vel(init_pos[0], init_pos[1], init_pos[2], init_vel[0], init_vel[1], init_vel[2])

orbit = plt.Circle((0, 0), r_orbit * u.to_Kpc, color='black', fill=False, linestyle='--', label='Star Orbit')
fig, ax = plt.subplots() 

ax.add_patch(orbit)
ax.set_xlim(-0.5 , 0.5)
ax.set_ylim(-0.5 , 0.5)

ax.set_aspect('equal', adjustable='box')

plt.scatter(r_orbit * u.to_Kpc, 0, color='red', label='Star Initial Position')
plt.scatter(0, 0, color='blue', label='Halo Center', marker='x')
plt.quiver(r_orbit * u.to_Kpc, 0, 0, init_vel[1] * u.to_kms, label='Star Initial Velocity', angles='xy', scale_units='xy', scale=10, color='green')

ax.set_xlabel(r"$x \;\;\mathrm{[kpc]}$", fontsize = 15)
ax.set_ylabel(r"$y \;\;\mathrm{[kpc]}$", fontsize = 15)
plt.show()


### Convergence Plot

In [ ]:

total_evolve_time = 5 * u.from_Gyr
num_steps = np.linspace(10, 100000, dtype=int, num = 10000)
dt = total_evolve_time / num_steps

print(num_steps)

vr_perc_at_dt= defaultdict(list)


for j in range(len(num_steps)):

    r_pos = init_pos
    v = init_vel
    velocities = [init_vel_sph]
    avg_r = init_pos_sph[0]

    r_pos_dt, v_dt, vel_disp, avg_r, r_mag, velocities = SSF.Time_step_t_indep(r_pos, v, dt[j], acc_mag, velocities, avg_r, i = 1)
    r_pos = r_pos_dt
    v = v_dt

    v_r_perc = velocities[-1][0] / np.linalg.norm(velocities[-1])

    vr_perc_at_dt[num_steps[j]].append(v_r_perc)

plt.plot(list(vr_perc_at_dt.keys()), list(vr_perc_at_dt.values()), lw=2)
plt.xlabel(r'Number of Time Steps', fontsize=15)
plt.ylabel(r'$v_{r} / |v|$', fontsize=15)
plt.yscale('log')
plt.xscale('log')
plt.legend()
plt.show()

log_num_steps = np.log10(list(vr_perc_at_dt.keys()))
log_vr_perc = np.log10(list(vr_perc_at_dt.values()))


slope, intercept = np.polyfit(log_num_steps, log_vr_perc, 1)

print("Slope:", slope)
print("Intercept:", intercept)

coefficient = 10**intercept

print("Coefficient:", coefficient)

plt.figure(figsize=(2, 1), dpi=150)
plt.text(0.1, 0.5, r'$v_r / |v| = %.2f N^{%.0f}$' % (coefficient, slope), fontsize=15)
plt.axis('off')
plt.show()



### Leap-frog Integrator Convergence plot

In [ ]:

total_evolve_time = 5 * u.from_Gyr
num_steps = np.linspace(10, 100000, dtype=int, num = 10000)
dt = total_evolve_time / num_steps

print(num_steps)

vr_perc_at_dt= defaultdict(list)


for j in range(len(num_steps)):

    r_pos = init_pos
    v = init_vel
    velocities = [init_vel_sph]
    avg_r = init_pos_sph[0]

    r_pos_dt, v_dt, vel_disp, avg_r, r_mag, velocities = SSF.Time_step_t_indep_leapfrog(r_pos, v, dt[j], acc_mag, velocities, avg_r, i = 1)
    r_pos = r_pos_dt
    v = v_dt

    v_r_perc = velocities[-1][0] / np.linalg.norm(velocities[-1])

    vr_perc_at_dt[num_steps[j]].append(v_r_perc)

plt.plot(list(vr_perc_at_dt.keys()), list(vr_perc_at_dt.values()), lw=2)
plt.xlabel(r'Number of Time Steps', fontsize=15)
plt.ylabel(r'$v_{r} / |v|$', fontsize=15)
plt.yscale('log')
plt.xscale('log')
plt.legend()
plt.show()

log_num_steps = np.log10(list(vr_perc_at_dt.keys()))
log_vr_perc = np.log10(list(vr_perc_at_dt.values()))


slope, intercept = np.polyfit(log_num_steps, log_vr_perc, 1)

print("Slope:", slope)
print("Intercept:", intercept)

coefficient = 10**intercept

print("Coefficient:", coefficient)

plt.figure(figsize=(2, 1), dpi=150)
plt.text(0.1, 0.5, r'$v_r / |v| = %.2f N^{%.0f}$' % (coefficient, slope), fontsize=15)
plt.axis('off')
plt.show()



### Hanno Reins Integrator Convergence plot

In [ ]:
import Stellar_sim_funcs as SSF
import importlib
importlib.reload(SSF)


total_evolve_time = 5 * u.from_Gyr
num_steps = np.linspace(10, 100000, dtype=int, num = 10000)
dt = total_evolve_time / num_steps

print(num_steps)

vr_perc_at_dt= defaultdict(list)


for j in range(len(num_steps)):

    r_pos = init_pos
    v = init_vel
    velocities = [init_vel_sph]
    avg_r = init_pos_sph[0]

    r_pos_dt, v_dt, vel_disp, avg_r, r_mag, velocities = SSF.Time_step_t_indep_Hanno_reins(r_pos, v, dt[j], acc_mag, velocities, avg_r, i = 1)
    r_pos = r_pos_dt
    v = v_dt

    v_r_perc = velocities[-1][0] / np.linalg.norm(velocities[-1])

    vr_perc_at_dt[num_steps[j]].append(v_r_perc)


mask = np.array(list(vr_perc_at_dt.values())).flatten() > 0

vr_perc_at_dt_values = np.array(list(vr_perc_at_dt.values()))[mask]
vr_perc_at_dt_keys = np.array(list(vr_perc_at_dt.keys()))[mask]

plt.plot(list(vr_perc_at_dt_keys), list(vr_perc_at_dt_values), lw=2)
plt.xlabel(r'Number of Time Steps', fontsize=15)
plt.ylabel(r'$v_{r} / |v|$', fontsize=15)
plt.yscale('log')
plt.xscale('log')
plt.legend()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# if you already have spherical versions, use those instead:
init_pos_sph = SSF.Cartesian_to_sph(*init_pos)
init_vel_sph = SSF.Cartesian_to_sph_vel(*init_pos, *init_vel)

# --- integration parameters ---
total_evolve_time = 5 * u.from_Gyr
num_steps = 10000
dt = total_evolve_time / num_steps

r_pos   = init_pos.copy()
v       = init_vel.copy()
avg_r   = init_pos_sph[0]
velocities = [init_vel_sph]

xs, ys = [], []   # store orbit in the orbital plane

for i in range(num_steps):
    xs.append(r_pos[0])
    ys.append(r_pos[1])

    r_pos, v, vel_disp, avg_r, r_mag, velocities = SSF.Time_step_t_indep_Hanno_reins(
        r_pos, v, dt, acc_mag, velocities, avg_r, i
    )

xs = np.array(xs)
ys = np.array(ys)

# --- plot the orbit ---
plt.figure(figsize=(5, 5))
plt.plot(xs * u.to_Kpc, ys * u.to_Kpc, lw=1)

# mark origin and starting point
plt.scatter(0.0, 0.0, marker='*', s=80, label='centre')
plt.scatter(xs[0] * u.to_Kpc, ys[0] * u.to_Kpc, marker='o', s=30, label='start')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Orbit in the xy plane')
plt.axis('equal')        # so orbit isn’t squashed
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 0.5)
plt.ylim(-0.5, 0.5)
plt.legend()
plt.tight_layout()
plt.show()
